# Análisis exploratorio de datos (EDA)

## Detección de sitios web de phishing mediante características de URL

Este notebook reproduce, paso a paso, el análisis exploratorio del archivo `Dataset.csv`.
El dataset original **no se modifica** en ninguna etapa. Las tablas y figuras se guardan
en `results/tables/` y `figures/`, respectivamente.

**Alcance:** observación, medición y documentación. No se eliminan registros, no se cambian
etiquetas y no se entrenan modelos.

## 1. Configuración y rutas

Se localiza la raíz del proyecto a partir del directorio de ejecución y se importa el módulo
`src/eda.py`, que contiene las funciones reproducibles del análisis.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "Dataset.csv").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("No se encontró Dataset.csv en la ruta actual ni en sus padres.")

sys.path.insert(0, str(ROOT / "src"))
import eda

DATA = ROOT / "Dataset.csv"
FIGURES = ROOT / "figures"
TABLES = ROOT / "results" / "tables"

print(f"Raíz del proyecto: {ROOT}")
print(f"Dataset: {DATA}")

## 2. Carga y dimensiones

Se lee el CSV con `pandas` y se comprueba la forma del conjunto de datos.

In [ ]:
df = eda.load_data(DATA)
print(f"Filas: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
df.head(3)

## 3. Tipos de datos y valores faltantes

La siguiente tabla resume el tipo inferido por `pandas`, los valores no nulos, los faltantes,
la cantidad de valores únicos y un ejemplo por columna.

In [ ]:
dimensiones = eda.build_dimension_table(df)
tipos = eda.build_type_table(df)
faltantes = eda.build_missing_table(df)

print(dimensiones.to_string(index=False))
print()
print(tipos.to_string(index=False))
print()
print(faltantes.to_string(index=False))

## 4. Distribución de la variable objetivo

`label` es la variable objetivo. Se muestra la distribución absoluta y porcentual. La
interpretación de `0` como legítimo y `1` como phishing es una hipótesis pendiente de
confirmar con la documentación de la fuente.

In [ ]:
etiquetas = eda.build_label_table(df)
print(etiquetas.to_string(index=False))
print(f"Suma de porcentajes: {etiquetas['porcentaje'].sum():.6f} %")

## 5. Estadísticas descriptivas

Se calculan medidas de tendencia central, dispersión, percentiles, asimetría y curtosis para
las 22 variables numéricas predictoras. `label` se excluye de esta tabla.

In [ ]:
descriptivas = eda.build_descriptive_table(df)
columnas_resumen = [
    "variable", "media", "mediana", "desv_est", "min", "max", "asimetria", "curtosis"
]
print(descriptivas[columnas_resumen].to_string(index=False))

## 6. Duplicados y repeticiones

Se distinguen tres conceptos:

1. **Filas completas duplicadas:** todas las columnas coinciden.
2. **URL repetidas:** la misma cadena de URL aparece más de una vez.
3. **Dominios repetidos:** muchas URL distintas comparten un dominio registrable.

Los dominios repetidos no se interpretan como filas duplicadas.

In [ ]:
duplicados = eda.build_duplicate_table(df)
print(duplicados.to_string(index=False))
print()
print(f"URL únicas: {df['url'].nunique(dropna=False):,}")
print(f"Dominios únicos: {df['dom'].nunique(dropna=False):,}")

## 7. Distribución de etiquetas por dominio

Se construye una tabla por dominio con el número de registros de cada clase y los dominios
que contienen ambas etiquetas. La tabla completa se guarda en
`results/tables/07_distribucion_etiquetas_por_dominio.csv`.

In [ ]:
dominios, resumen_dominios, contradictorios, top_dominios = eda.build_domain_tables(df)

print(resumen_dominios.to_string(index=False))
print()
print(f"Dominios contradictorios: {len(contradictorios)}")
print(contradictorios.to_string(index=False))
print()
print(eda.build_domain_normalization_table(df).to_string(index=False))

## 8. Variables constantes, valores extremos y asimetría

Se marcan como casi constantes las columnas cuya categoría más frecuente concentra al menos
el 99 % de los registros. Los valores extremos se describen con el criterio de rango
intercuartílico (1,5 × IQR), sin eliminarlos.

In [ ]:
constantes = eda.build_constant_table(df)
extremos = eda.build_outlier_table(df)

print(constantes.to_string(index=False))
print()
columnas_extremos = [
    "variable", "asimetria", "porcentaje_atipicos_iqr", "porcentaje_ceros", "asimetria_alta"
]
print(extremos[columnas_extremos].to_string(index=False))

## 9. Correlaciones

Se calcula la matriz de correlación de Pearson entre las variables numéricas. También se
identifican pares con correlación absoluta mayor o igual a 0,90 y la asociación lineal de
cada variable con `label`. La correlación no implica causalidad.

In [ ]:
matriz_correlacion, pares_altos, correlacion_label = eda.build_correlation_tables(df)

print("Pares con |r| >= 0,90:")
print(pares_altos.to_string(index=False))
print()
print("Variables con mayor asociación lineal con label:")
print(correlacion_label.head(12).to_string(index=False))

## 10. Comparación de características entre clases

Se comparan las variables numéricas entre las clases mediante medias, medianas, la prueba de
Mann-Whitney y el efecto de rango biserial. Con 116 600 observaciones, valores muy pequeños de
`p` no implican necesariamente una diferencia de magnitud relevante; por eso se reporta el
tamaño del efecto.

In [ ]:
comparacion = eda.build_class_comparison(df)
columnas_comparacion = [
    "variable", "media_clase_0", "media_clase_1", "mediana_clase_0",
    "mediana_clase_1", "p_valor", "rank_biserial"
]
print(comparacion[columnas_comparacion].to_string(index=False))
print()
print("Variables categóricas por clase:")
print(eda.build_categorical_frequencies(df).to_string(index=False))

## 11. Generación reproducible de tablas y figuras

La siguiente llamada reproduce todas las tablas y figuras del análisis y ejecuta los controles
de calidad: suma de clases, porcentajes, faltantes, dominios y existencia de archivos.

In [ ]:
resumen = eda.run_eda(DATA, FIGURES, TABLES)
print(f"Filas: {resumen['filas']:,}")
print(f"Columnas: {resumen['columnas']}")
print(f"Dominios contradictorios: {resumen['dominios_contradictorios']}")
print(f"Verificaciones: {resumen['verificaciones']}")

## 12. Figuras generadas

Las figuras se guardan en `figures/` y se muestran a continuación. Si el notebook se ejecuta
desde otra ubicación, las rutas relativas pueden no renderizar las imágenes, pero los archivos
se generan en la carpeta configurada.

### Figura 1. Distribución de clases

![Distribución de clases](../figures/fig01_distribucion_clases.png)

### Figura 2. Valores faltantes por columna

![Valores faltantes](../figures/fig02_valores_faltantes.png)

### Figura 3. Distribuciones numéricas relevantes

![Distribuciones numéricas](../figures/fig03_distribuciones_numericas.png)

### Figura 4. Comparación de características por clase

![Comparación por clase](../figures/fig04_comparacion_clases.png)

### Figura 5. Matriz de correlación

![Matriz de correlación](../figures/fig05_matriz_correlacion.png)

### Figura 6. Dominios con etiquetas contradictorias

![Dominios contradictorios](../figures/fig06_dominios_contradictorios.png)

In [ ]:
for path in sorted(FIGURES.glob("*.png")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")
for path in sorted(TABLES.glob("*.csv")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

## 13. Observaciones para la Fase 3

- La distribución de clases está desbalanceada (85,76 % / 14,24 %), por lo que la exactitud
  por sí sola no será una métrica adecuada.
- Existen 1 369 filas duplicadas exactas en 1 007 grupos de URL. Su tratamiento debe decidirse
  antes de la partición para evitar fuga de información.
- Hay 24 dominios con ambas etiquetas; los servicios de alojamiento y las URL acortadas
  concentran una parte importante de los casos de la clase 1.
- `is_ip` es casi constante: solo 13 registros tienen valor 1.
- Varias variables presentan asimetría alta y valores extremos; no deben eliminarse sin un
  análisis y una justificación.
- `url_len` y `letter_cnt` tienen una correlación de 0,972; `eq_cnt` y `amp_cnt`, de 0,928.
  Esto sugiere redundancia que debe evaluarse en el preprocesamiento.
- La diferencia de `is_https` entre clases (35,1 % en clase 0 frente a 91,4 % en clase 1) es
  un hallazgo descriptivo que podría reflejar diferencias en la recolección de datos; no debe
  interpretarse como una relación causal.
- La semántica exacta de las etiquetas y la procedencia del dataset deben documentarse antes
  de presentar conclusiones definitivas.

El informe completo del EDA se encuentra en `results/eda_summary.md`. Esta fase se detiene
aquí: no se realiza partición, limpieza ni entrenamiento.